# Notebook 4: De RAG a chatbot

**ML2 — Semana 7** | Escuela de Gobierno, UAI | Prof. Naim Bro

---

## De la consulta aislada al chatbot

```
documentos  →  chunking  →  embeddings  →  retrieval  →  contexto  →  respuesta  →  historial  →  interfaz
✓ NB1          ✓ NB2        ✓ NB2          ✓ NB2         ✓ NB3         ✓ NB3         ▲              ▲
                                                                               ESTAMOS AQUÍ
```

En el notebook anterior armamos un **RAG one-shot**: una pregunta entra, una respuesta sale.
Eso sirve para consultas aisladas, pero no es lo que la gente espera cuando abre un chatbot.

En un chatbot la conversación **acumula contexto**: la segunda pregunta depende de la primera,
y el sistema tiene que recordar de qué se está hablando.

**Idea clave del notebook:**

> Un chatbot no es una nueva especie de sistema.
> Es un pipeline de RAG + historial conversacional + interfaz de chat.

## 0) Qué ya sabemos y qué haremos ahora

Ya sabemos (NB1–NB3):
- partir de un corpus y **chunkear** documentos,
- generar **embeddings** y buscar por **similitud semántica** (retrieval),
- armar un **contexto** con los chunks recuperados,
- generar una **respuesta grounded** con un LLM,
- detectar fallas típicas de RAG (chunking pobre, queries ambiguas, fuera de corpus, alucinaciones).

En este notebook agregamos **dos piezas nuevas**:

1. **Historial conversacional**: el chatbot recuerda lo último que se habló para entender preguntas de seguimiento.
2. **Interfaz de chat**: una UI mínima con `gradio.ChatInterface` que corre en Colab y se puede compartir.

Todo lo demás lo reusamos del NB3 (sin reexplicar embeddings ni chunking desde cero).

## 1) Instalación

Agregamos `gradio` al stack que ya veníamos usando.

In [ ]:
!pip install -q gdown google-genai gradio

## 2) Preparar el pipeline base

Esta sección es el **pipeline de RAG del NB3 condensado**: API key, corpus, parse, chunking, embeddings y retrieval.
Si algo acá no te calza, vuelve al NB3 — allí está la explicación larga. Aquí solo lo dejamos listo para chatear.

In [ ]:
import json
import os
import re
import zipfile
import shutil
from pathlib import Path

import numpy as np
import gdown
from google import genai
from google.genai import types

# --- API key ---
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
if api_key is None and Path("config_gemini.json").exists():
    with open("config_gemini.json") as f:
        api_key = json.load(f)["api_key"]
if api_key is None:
    api_key = os.getenv("GEMINI_API_KEY")
if api_key is None:
    raise ValueError("No encontré la API key.")
client = genai.Client(api_key=api_key)
print("Cliente Gemini listo.")

In [ ]:
# --- Corpus ---
CORPUS_URL = "https://drive.google.com/file/d/1T6W_Nsi37p5qJfQP4WaBtximWh_joWKM/view?usp=sharing"
ZIP_PATH = Path("rag_txt_corpus_semana3.zip")
EXTRACT_DIR = Path("rag_txt_corpus_semana3")

if not ZIP_PATH.exists():
    gdown.download(CORPUS_URL, str(ZIP_PATH), fuzzy=True, quiet=False)
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)
txt_files = sorted([p for p in EXTRACT_DIR.glob("*.txt") if p.name != "README.txt"])

# --- Parse ---
HEADER_MAP = {"TÍTULO": "titulo", "INSTITUCIÓN": "institucion", "FECHA": "fecha", "TIPO": "tipo"}
def parse_document(path):
    raw = path.read_text(encoding="utf-8").strip()
    lines = raw.splitlines()
    meta = {"filename": path.name, "titulo": None, "institucion": None, "fecha": None, "tipo": None}
    body_lines, header_done = [], False
    for line in lines:
        clean = line.strip()
        if not header_done and ":" in clean:
            key, value = clean.split(":", 1)
            if key.strip() in HEADER_MAP:
                meta[HEADER_MAP[key.strip()]] = value.strip()
                continue
        header_done = True
        body_lines.append(line)
    meta["texto"] = "\n".join(body_lines).strip()
    return meta
documents = [parse_document(p) for p in txt_files]

# --- Chunking ---
def split_paragraphs(text):
    text = text.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text).strip()
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

def chunk_paragraphs(paragraphs, max_chars=900, overlap_paragraphs=1):
    chunks, current, current_len = [], [], 0
    for para in paragraphs:
        if current and current_len + len(para) + 2 > max_chars:
            chunks.append("\n\n".join(current).strip())
            current = current[-overlap_paragraphs:] if overlap_paragraphs > 0 else []
            current_len = sum(len(p) for p in current)
        current.append(para)
        current_len += len(para) + 2
    if current:
        chunks.append("\n\n".join(current).strip())
    return chunks

all_chunks = []
for doc in documents:
    for i, ct in enumerate(chunk_paragraphs(split_paragraphs(doc["texto"])), 1):
        all_chunks.append({
            "chunk_id": f"{doc['filename']}::chunk_{i}",
            "filename": doc["filename"], "titulo": doc["titulo"],
            "institucion": doc["institucion"], "chunk_index": i,
            "chunk_text": ct,
            "text_for_embedding": f"Título: {doc['titulo']}\nInstitución: {doc['institucion']}\n\n{ct}",
        })
print(f"{len(documents)} documentos → {len(all_chunks)} chunks")

In [ ]:
# --- Embeddings ---
def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT", batch_size=20):
    all_emb = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        result = client.models.embed_content(
            model="gemini-embedding-001", contents=batch,
            config=types.EmbedContentConfig(task_type=task_type),
        )
        all_emb.extend([np.array(e.values, dtype=np.float32) for e in result.embeddings])
        print(f"  Embeddings: {min(start + batch_size, len(texts))}/{len(texts)}")
    return np.vstack(all_emb)

print("Generando embeddings...")
corpus_embeddings = embed_texts([c["text_for_embedding"] for c in all_chunks])
print(f"Listo: {corpus_embeddings.shape}")

# --- Retrieval ---
def embed_query(query):
    result = client.models.embed_content(
        model="gemini-embedding-001", contents=[query],
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    return np.array(result.embeddings[0].values, dtype=np.float32)

def search(query, top_k=3):
    scores = corpus_embeddings @ embed_query(query)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{
        "rank": rank, "score": float(scores[idx]),
        "filename": all_chunks[idx]["filename"], "titulo": all_chunks[idx]["titulo"],
        "chunk_index": all_chunks[idx]["chunk_index"], "chunk_text": all_chunks[idx]["chunk_text"],
    } for rank, idx in enumerate(ranked, 1)]

def build_context(results):
    """Arma un bloque de contexto a partir de los chunks recuperados."""
    blocks = []
    for r in results:
        blocks.append(
            f"[FUENTE: {r['filename']} | {r['titulo']} | chunk {r['chunk_index']}]\n"
            f"{r['chunk_text']}"
        )
    return ("\n\n" + "=" * 80 + "\n\n").join(blocks)

print("\nPipeline de retrieval listo.")

## 3) De respuesta RAG a respuesta conversacional

Este es el **corazón** del notebook.

Una consulta one-shot es autosuficiente: la pregunta tiene todo lo necesario para buscar en el corpus.
En una conversación real eso deja de ser cierto. Mira esta secuencia:

> **Usuario:** ¿Qué documentos se piden para renovar el permiso de feria libre?
> **Bot:** [responde sobre el trámite]
> **Usuario:** ¿Y quién lo autoriza?

La tercera frase, sola, es **inútil** para el motor de retrieval: no dice *qué* se autoriza.
El chatbot necesita leerla **junto con** el historial reciente para entender que "lo" = "el permiso de feria libre".

Para lograrlo, hacemos dos cosas:

1. **Formatear el historial** reciente como texto plano que el LLM pueda leer.
2. **Inyectar ese historial** al prompt, de modo que el modelo responda en contexto.

Para el **retrieval** también vamos a usar el historial: si la query del turno actual es muy corta o ambigua, la combinamos con los turnos anteriores antes de buscar. Es una heurística simple, pero cambia bastante el comportamiento del sistema.

In [ ]:
def format_history(history, max_turns=4):
    """Convierte el historial en texto legible para el LLM.

    `history` es una lista de pares (user_msg, bot_msg) — el formato que usa gradio.ChatInterface.
    Nos quedamos solo con los últimos `max_turns` turnos para no inflar el prompt.
    """
    if not history:
        return ""
    recent = history[-max_turns:]
    lines = []
    for user_msg, bot_msg in recent:
        if user_msg:
            lines.append(f"Usuario: {user_msg}")
        if bot_msg:
            lines.append(f"Asistente: {bot_msg}")
    return "\n".join(lines)


def build_retrieval_query(message, history, max_turns=2):
    """Combina el mensaje actual con el historial reciente para mejorar el retrieval.

    Si la pregunta es corta o ambigua ("¿y quién?", "¿desde cuándo?"),
    el contexto previo ayuda a que los embeddings apunten al tema correcto.
    """
    if not history:
        return message
    recent = history[-max_turns:]
    pieces = []
    for user_msg, _ in recent:
        if user_msg:
            pieces.append(user_msg)
    pieces.append(message)
    return " ".join(pieces)

In [ ]:
def chat_with_rag(message, history, top_k=3, model="gemini-2.5-flash"):
    """Pipeline conversacional: retrieval con contexto + respuesta grounded con historial.

    Devuelve un dict con la respuesta y los chunks usados, para poder inspeccionar.
    """
    # 1) Query enriquecida con historial (ayuda al retrieval en preguntas de seguimiento).
    retrieval_query = build_retrieval_query(message, history)
    retrieved = search(retrieval_query, top_k=top_k)
    context = build_context(retrieved)

    # 2) Historial conversacional para que el modelo entienda referencias ("eso", "lo", "ahí").
    history_text = format_history(history)
    history_block = f"\nHistorial reciente de la conversación:\n{history_text}\n" if history_text else ""

    # 3) Prompt del chatbot. Exige grounding, incertidumbre explícita y fuentes.
    prompt = (
        "Eres un asistente conversacional que responde preguntas sobre documentos del sector público.\n\n"
        "Reglas:\n"
        "1. Responde solo con información del CONTEXTO recuperado.\n"
        "2. Si la evidencia es insuficiente, dilo explícitamente ('No tengo información suficiente en el corpus para responder eso').\n"
        "3. No inventes datos, cifras, fechas ni nombres.\n"
        "4. Usa el HISTORIAL solo para entender a qué se refiere el usuario (referencias, seguimientos), NO como fuente de hechos.\n"
        "5. Sé breve y claro.\n"
        "6. Termina siempre con una línea 'Fuentes:' listando los archivos usados.\n"
        f"{history_block}\n"
        f"Pregunta actual del usuario:\n{message}\n\n"
        f"Contexto recuperado:\n{context}"
    )

    response = client.models.generate_content(model=model, contents=prompt)
    return {
        "answer": response.text,
        "retrieved": retrieved,
        "retrieval_query": retrieval_query,
    }

## 4) Primera versión del chatbot en Python puro

Antes de montar una interfaz, conviene ver la conversación en texto plano.
Así queda claro que el "chatbot" no es magia: es solo `chat_with_rag` llamada varias veces,
acumulando turnos en una lista.

Simulamos tres mensajes encadenados:

1. una pregunta concreta sobre un trámite,
2. una **pregunta de seguimiento** que solo se entiende con el historial,
3. una pregunta **fuera del corpus**, para ver si el sistema declara su límite.

In [ ]:
history = []  # lista de pares (user_msg, bot_msg)

turnos = [
    "¿Qué documentos se piden para renovar el permiso de feria libre?",
    "¿Y quién lo autoriza?",                       # seguimiento: depende del turno anterior
    "¿Qué dice la ley de protección de datos sobre esto?",  # probablemente fuera del corpus
]

for i, mensaje in enumerate(turnos, 1):
    print(f"\n{'=' * 80}\nTURNO {i} — Usuario: {mensaje}\n{'=' * 80}")
    resultado = chat_with_rag(mensaje, history, top_k=3)
    print(f"[query usada para retrieval] {resultado['retrieval_query']}")
    print(f"[top chunks] {[r['filename'] for r in resultado['retrieved']]}")
    print(f"\nAsistente:\n{resultado['answer']}")
    history.append((mensaje, resultado["answer"]))

## 5) Comparar: sin historial vs. con historial

Para ver qué agrega realmente un chatbot, corremos la misma pregunta de seguimiento en dos modos:

- **Sin historial**: el sistema recibe solo la frase "¿Y quién lo autoriza?". El retrieval no sabe qué buscar.
- **Con historial**: el sistema recibe la frase más los turnos previos. Ahora "lo" queda anclado al tema.

Este contraste es la razón principal por la que un chatbot se siente útil y no frustrante.

In [ ]:
pregunta_inicial = "¿Qué documentos se piden para renovar el permiso de feria libre?"
pregunta_seguimiento = "¿Y quién lo autoriza?"

# Simulamos un historial donde ya hubo la primera pregunta y respuesta.
primer_turno = chat_with_rag(pregunta_inicial, history=[], top_k=3)
historial_simulado = [(pregunta_inicial, primer_turno["answer"])]

print("=" * 80)
print("SIN HISTORIAL — la frase '¿Y quién lo autoriza?' va sola")
print("=" * 80)
sin_hist = chat_with_rag(pregunta_seguimiento, history=[], top_k=3)
print(f"[query retrieval] {sin_hist['retrieval_query']}")
print(f"[top chunks]     {[r['filename'] for r in sin_hist['retrieved']]}")
print(f"\nRespuesta:\n{sin_hist['answer']}")

print("\n" + "=" * 80)
print("CON HISTORIAL — la frase va junto con el turno anterior")
print("=" * 80)
con_hist = chat_with_rag(pregunta_seguimiento, history=historial_simulado, top_k=3)
print(f"[query retrieval] {con_hist['retrieval_query']}")
print(f"[top chunks]     {[r['filename'] for r in con_hist['retrieved']]}")
print(f"\nRespuesta:\n{con_hist['answer']}")

**Lectura del experimento.**
La versión sin historial suele recuperar chunks desalineados (porque "¿y quién lo autoriza?" es genérica)
y responde con evasivas o preguntando de qué hablamos. La versión con historial recupera chunks
del trámite correcto y responde de forma útil. La diferencia no está en el modelo, está en **qué texto entra al retrieval y al prompt**.

## 6) Construir la interfaz con Gradio

Ya tenemos la lógica. Ahora montamos la interfaz con `gr.ChatInterface`,
que es la forma más corta de tener un chatbot usable en Colab.

`ChatInterface` espera una función con firma `fn(message, history)` que devuelva un **string**.
Nuestra `chat_with_rag` devuelve un dict, así que escribimos un wrapper que:

- llama a `chat_with_rag`,
- toma la respuesta del modelo,
- le agrega una línea compacta de fuentes con los chunks recuperados (trazabilidad visible al usuario).

In [ ]:
import gradio as gr

def chatbot_fn(message, history):
    resultado = chat_with_rag(message, history, top_k=3)
    respuesta = resultado["answer"]

    fuentes = sorted({r["filename"] for r in resultado["retrieved"]})
    linea_fuentes = "\n\n_Chunks recuperados: " + ", ".join(fuentes) + "_"
    return respuesta + linea_fuentes


demo = gr.ChatInterface(
    fn=chatbot_fn,
    title="Chatbot RAG — corpus público (demo ML2)",
    description=(
        "Responde solo con documentos del corpus del curso. "
        "Si no hay evidencia, lo dice. Muestra los archivos que usó."
    ),
    examples=[
        "¿Qué documentos se piden para renovar el permiso de feria libre?",
        "¿Y quién lo autoriza?",
        "¿Cuándo se activa un albergue de invierno?",
        "¿Qué pasa si hay un corte de agua no programado?",
    ],
)

# share=True genera un link público temporal, útil para probar desde el celular o compartir en clase.
demo.launch(share=True, debug=False)

## 7) Qué hace útil a este chatbot y dónde falla

**Por qué puede ser útil en sector público**
- Responde **solo con documentos oficiales** del corpus, no con la memoria genérica del LLM.
- Cada respuesta **cita fuentes**, lo que permite auditar y ganar confianza.
- Declara **incertidumbre** cuando el corpus no contiene la respuesta.
- Mantiene una **conversación natural**, no obliga a repetir el contexto en cada pregunta.

**En qué se diferencia de solo "preguntarle a Gemini"**
- Gemini por sí solo responde con su conocimiento general — puede estar desactualizado o ser incorrecto para tu dominio.
- El chatbot RAG responde **anclado al corpus específico** de la institución.
- Tiene trazabilidad: puedes ver exactamente qué documento sustenta cada respuesta.

**Dónde falla (todavía)**
- **Mal retrieval**: si los embeddings no encuentran los chunks correctos, la respuesta será pobre aunque el modelo sea excelente.
- **Chunks pobres**: trozos cortados por la mitad de una idea pierden información clave.
- **Historial mal manejado**: si arrastramos demasiados turnos el prompt se infla y el modelo se confunde; si arrastramos muy pocos perdemos contexto de seguimiento.
- **Prompt deficiente**: si el prompt no exige grounding explícito, el modelo se relaja y mezcla su conocimiento general.
- **Preguntas fuera del corpus**: incluso con instrucciones claras, a veces el modelo intenta responder igual.
- **Alucinaciones residuales**: aun con buen contexto, el LLM puede "redondear" detalles que no están en los chunks.

## 8) Diseño responsable para sector público

Un chatbot con RAG en el Estado no es un juguete. Aunque el sistema sea técnicamente correcto,
las decisiones de diseño tienen consecuencias sobre ciudadanos reales. Esta checklist resume los
principios mínimos que deberían verificarse antes de poner un sistema así frente a usuarios:

- **Grounding estricto**: el chatbot responde solo con evidencia del corpus; si no la hay, lo declara.
- **Incertidumbre visible**: respuestas parciales o dudosas deben marcarse como tales, no disfrazarse de certeza.
- **Trazabilidad**: cada respuesta cita sus fuentes (archivo, sección, fecha si aplica).
- **Privacidad**: el corpus no debe incluir datos personales sensibles; si los incluye, se anonimizan.
- **Límites del rol**: el chatbot **informa y apoya**, no decide trámites, no niega beneficios, no reemplaza al funcionario.
- **Humano en el loop**: para decisiones con impacto (beneficios, sanciones, derivaciones), el chatbot sugiere, un humano resuelve.
- **Mensaje de alcance**: el usuario debe entender, antes de preguntar, qué cubre el sistema y qué no.
- **Monitoreo**: registrar consultas y respuestas (con los debidos resguardos) para detectar fallas, sesgos y preguntas frecuentes no cubiertas.

## 9) Ejercicios

Los ejercicios están pensados para que experimentes con el chatbot que acabas de construir
y lo conviertas en un prototipo para tu proyecto final.

### Ejercicio A — Cambiar el prompt del chatbot

Compara una versión **estricta** (sólo corpus, cita fuentes, declara incertidumbre) con una versión **laxa**
(puede complementar con conocimiento general). ¿Cuál prefieres para un servicio público? ¿Por qué?
¿Cambia tu respuesta si el servicio es informativo vs. si resuelve trámites?

In [ ]:
# Ejercicio A: prompt estricto vs. prompt laxo sobre la misma conversación

def chat_custom_prompt(message, history, instrucciones, top_k=3, model="gemini-2.5-flash"):
    retrieval_query = build_retrieval_query(message, history)
    retrieved = search(retrieval_query, top_k=top_k)
    context = build_context(retrieved)
    history_text = format_history(history)
    history_block = f"\nHistorial reciente:\n{history_text}\n" if history_text else ""

    prompt = (
        f"{instrucciones}\n"
        f"{history_block}\n"
        f"Pregunta actual del usuario:\n{message}\n\n"
        f"Contexto recuperado:\n{context}"
    )
    response = client.models.generate_content(model=model, contents=prompt)
    return response.text


pregunta = "¿Qué apoyo existe para jóvenes que buscan empleo?"

prompt_estricto = (
    "Responde solo con el contexto. Si falta evidencia, di que no tienes información suficiente. "
    "Cita fuentes al final. Sé breve."
)
prompt_laxo = (
    "Usa el contexto como apoyo pero puedes complementar con tu conocimiento general. "
    "Responde lo más completo posible."
)

print("=== PROMPT ESTRICTO ===")
print(chat_custom_prompt(pregunta, history=[], instrucciones=prompt_estricto))
print("\n=== PROMPT LAXO ===")
print(chat_custom_prompt(pregunta, history=[], instrucciones=prompt_laxo))

# Tu análisis:
# - ¿Cuál suena más confiable para un sitio de un municipio?
# - ¿Cuál genera más riesgo de alucinación?

### Ejercicio B — Comparar con y sin historial

Escoge un tema del corpus y arma una conversación de **3 turnos**, donde el segundo y tercero
sean preguntas de seguimiento ("¿y quién lo implementa?", "¿y desde cuándo?", "¿y qué pasa en ese caso?").
Corre los mismos tres turnos dos veces: una sin pasar `history`, otra pasando `history`.
¿En cuál el chatbot se siente "útil"? ¿En cuál parece no entender?

In [ ]:
# Ejercicio B: con historial vs. sin historial en una conversación de 3 turnos

turnos_ej = [
    # TODO: reemplaza por una secuencia relevante a tu corpus
    "¿Cómo se priorizan los hogares para una beca de conectividad?",
    "¿Y quién revisa las postulaciones?",
    "¿Qué pasa si faltan documentos?",
]

print("=" * 80)
print("SIN HISTORIAL (cada pregunta va sola)")
print("=" * 80)
for t in turnos_ej:
    r = chat_with_rag(t, history=[], top_k=3)
    print(f"\nUsuario: {t}\nAsistente: {r['answer']}\n")

print("=" * 80)
print("CON HISTORIAL (acumulamos turnos)")
print("=" * 80)
hist = []
for t in turnos_ej:
    r = chat_with_rag(t, history=hist, top_k=3)
    print(f"\nUsuario: {t}\nAsistente: {r['answer']}\n")
    hist.append((t, r["answer"]))

### Ejercicio C — Probar fallas

Probaste el camino feliz. Ahora rompe el sistema a propósito con tres tipos de query:

1. **Ambigua**: una pregunta con doble tema (p. ej. mezcla salud y vivienda).
2. **Fuera del corpus**: algo claramente no cubierto por los documentos.
3. **Parcialmente cubierta**: algo donde el corpus responde una parte pero no toda.

Para cada caso, anota qué pasa: ¿el sistema declara incertidumbre? ¿inventa? ¿mezcla?

In [ ]:
# Ejercicio C: fallas típicas

casos = {
    "ambigua":             "¿Cómo se manejan los reclamos sobre baches y problemas de salud?",
    "fuera del corpus":    "¿Cuál es la tasa de interés vigente del Banco Central?",
    "parcial":             "¿Qué apoyo existe para jóvenes que buscan empleo y vivienda?",
}

for etiqueta, q in casos.items():
    print(f"\n{'=' * 80}\n[{etiqueta.upper()}] {q}\n{'=' * 80}")
    r = chat_with_rag(q, history=[], top_k=3)
    print(f"[chunks] {[c['filename'] for c in r['retrieved']]}")
    print(f"\n{r['answer']}")

### Ejercicio D — Adaptar el chatbot al proyecto final

Sin código por ahora. Responde en 1 párrafo cada punto, pensando en el proyecto de tu grupo:

1. **Corpus**: ¿qué documentos alimentan el chatbot? ¿quién los mantiene actualizados?
2. **Usuarios**: ¿quién pregunta? ¿funcionarios internos, ciudadanía, ambos?
3. **Tipo de preguntas**: ¿informativas, procedimentales, de casos específicos?
4. **Tono de respuesta**: ¿formal institucional, cercano, técnico?
5. **Riesgos**: ¿qué pasa si el chatbot se equivoca? ¿quién asume la responsabilidad?
6. **Evaluación**: ¿cómo sabrás si funciona bien antes de ponerlo en producción?

In [ ]:
# Ejercicio D: boceto del chatbot de tu proyecto

# 1) Corpus:


# 2) Usuarios:


# 3) Tipo de preguntas:


# 4) Tono de respuesta:


# 5) Riesgos y mitigaciones:


# 6) Plan de evaluación:


### Ejercicio E — Mejorar la trazabilidad

En la interfaz actual mostramos los **nombres de archivo** usados. Eso es un mínimo razonable,
pero se puede hacer mejor. Modifica `chatbot_fn` para que la respuesta incluya, por ejemplo:

- el **score de similitud** del chunk top,
- un **extracto corto** del chunk más relevante (los primeros ~200 caracteres),
- o un **desplegable** con los chunks completos.

Si te quieres animar, separa el chunk-preview en un bloque visual distinto usando `gr.Blocks` en vez de `gr.ChatInterface`.

In [ ]:
# Ejercicio E: trazabilidad mejorada — reemplaza chatbot_fn y vuelve a lanzar demo

def chatbot_fn_v2(message, history):
    resultado = chat_with_rag(message, history, top_k=3)
    respuesta = resultado["answer"]

    # TODO: agrega aquí el formato de fuentes que prefieras.
    detalle_fuentes = "\n\n---\n**Fuentes detalladas:**"
    for r in resultado["retrieved"]:
        extracto = r["chunk_text"][:200].replace("\n", " ")
        detalle_fuentes += (
            f"\n- `{r['filename']}` · chunk {r['chunk_index']} · score={r['score']:.3f}"
            f"\n  _{extracto}..._"
        )
    return respuesta + detalle_fuentes

# Para probar:
# demo_v2 = gr.ChatInterface(fn=chatbot_fn_v2, title="Chatbot RAG v2 — con trazabilidad")
# demo_v2.launch(share=True)

## 10) Cómo escalar esto después

Este notebook es suficiente para un **prototipo de curso** y para presentar el proyecto final.
Para llevarlo a una versión real de servicio público, hay varias direcciones posibles:

- **Corpus**: reemplazar los documentos de ejemplo por documentos oficiales actualizados, con un proceso claro de mantención.
- **Almacenamiento vectorial**: usar una base vectorial (Chroma, FAISS, pgvector) para no recalcular embeddings en cada sesión.
- **Evaluación sistemática**: conjunto fijo de preguntas con respuestas esperadas, métricas de recall del retrieval y de fidelidad de la respuesta.
- **Logging y monitoreo**: registrar consultas, chunks recuperados y respuestas (con resguardos de privacidad) para mejorar el sistema con el uso real.
- **Interfaz más robusta**: Streamlit o una web app conectada a autenticación institucional.
- **Deployment**: cloud del organismo, con control de versiones del corpus, de los prompts y del modelo.

Todo eso es importante, pero **ninguna de esas piezas cambia la lógica central**
que acabamos de ver: *retrieval + contexto + generación + historial + interfaz*.

## Resumen de la secuencia completa

| Notebook | Foco | Aprendizaje clave |
|----------|------|-------------------|
| NB1 | Corpus y documentos | Entender los datos antes de procesarlos |
| NB2 | Chunking y retrieval | Buscar por significado, no por palabras |
| NB3 | RAG completo | Generar respuestas fundamentadas en evidencia |
| NB4 | De RAG a chatbot | Agregar historial e interfaz sin perder grounding |

```
documentos  →  chunking  →  embeddings  →  retrieval  →  contexto  →  respuesta  →  historial  →  interfaz
✓ NB1          ✓ NB2        ✓ NB2          ✓ NB2         ✓ NB3         ✓ NB3         ✓ NB4         ✓ NB4
```